### 1. Introduction
DS 2002: Data Science Systems, 3/23/2026, Dino Papalabrakopoulos
The following notebook contains the all code and accompanying comments for Project 1. Please refer to the accompanying documentation for a description of the database-building process and deployment strategy.

**An important note:** I will repeat some data retrievals for the sake of demonstrating the process more clearly.

### 2. Environment Setup

Ensure that the MySQL80 service is running.

#### 2.1. Import Dependencies

All packages necessary for this notebook are imported in the following cell. Any package not loaded onto the local system may be installed by **python -m pip install [package_name]** on the console.

In [1]:
#import dependencies
import os
import json
import numpy as np
import datetime
import certifi
import pandas as pd
import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text


#### 2.2. Define Global Variables

All reused variables are defined in the following cell. Change the login and connection variables as needed.

In [2]:
#common global variables
dst_dbname = 'adventureworks_dw'

#MySQL-specific global variables
user_id = 'root'
pwd = 'Athens#1!'
host_name = 'localhost'
port = '3306'
src_dbname = 'adventureworks'

#MongoDB-specific global variables
mongo_user_name = 'dino_papas'
mongo_pwd = 'Athens#1!'
cluster_name = 'ds2002'
cluster_subnet = 'fbtvvuj'
cluster_location = 'atlas'
mongo_db_name = 'adventure_works_purchases_and_sales'



#### 2.3. Define Functions for Retrieving Data From and Setting Data Into Databases

All reused functions are defined in the following cell.

In [3]:

"""
Name --> create_and_use_database
Purpose --> connects to a local MySQL server and creates a database of the specified name
    user_id: text string, the user id of the MySQL server
    pwd: text string, the password corresponding to user_id on the MySQL server
    host_name: text string, the host name of the MySQL server
    db_name: text string, the desired name of the new database
Returns --> nothing
"""
def create_and_use_database(user_id, pwd, host_name, db_name):
    #establish the connection to the MySQL server
    connection_string = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
    sql_engine = create_engine(connection_string,pool_recycle = 3600)
    connection = sql_engine.connect()

    connection.execute(text(f"DROP DATABASE IF EXISTS `{db_name}`;")) #SQL statement to check if the database exists and drop it if so
    connection.execute(text(f"CREATE DATABASE `{db_name}`;")) #SQL statement to create a new database
    connection.execute(text(f"USE `{db_name}`;")) #SQL statement to direct SQL server to use new database

    connection.close()

"""
Name --> get_dataframe
Purpose --> connects to the local MySQL server and attempts to execute a passed query
    user_id: text string, the user id of the MySQL server
    pwd: text string, the password corresponding to user_id on the MySQL server
    host_name: text string, the host name of the MySQL server
    db_name: text string, the name of the database to Extract from
    query: text string, the desired query to run
Returns --> returns the dataframe corresponding to the data specified by the query
    Note: error will occur if the query is constructed incorrectly or references objects that do not exist
"""
def get_dataframe(user_id, pwd, host_name, db_name, query):
    #establish the connection to the MySQL server
    connection_string  = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sql_engine = create_engine(connection_string, pool_recycle=3600)
    connection = sql_engine.connect()

    df = pd.read_sql(query, connection) #retrieve data according to the SQL query and format as DataFrame
    connection.close()
    
    return df #return data in DataFrame format

"""
Name --> set_dataframe
Purpose --> connects to the local MySQL database and attempts to create a new table or append to that table depending on the passed operation
    user_id: text string, the user id of the MySQL server
    pwd: text string, the password corresponding to user_id on the MySQL server
    host_name: text string, the host name of the MySQL server
    db_name: text string, the name of the database to push to
    df: DataFrame, the data to push to the database
    table_name: text string, what to name the table in the database
    pk_column: text string, what to name the AUTO_INCREMENT primary key in the event of an insert operation without a pre-existing table
    db_operation: text_string (insert or update), if update, attempts to append df to table with table_name; if insert and table with table_name exists, replace the table;
        if insert and table with table_name does not exist, create a new table.
Returns --> nothing
"""
def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    #establish the connection to the MySQL server
    connection_string = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sql_engine = create_engine(connection_string, pool_recycle=3600)
    db_connection = sql_engine.connect()
    
    if db_operation in ['insert', 'update']: # if the operation is either insert or update
        if db_operation.lower() == "insert": # if the operation is insert
            # push the df with name table_name to the database, replacing an existing table with the same name
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            # create an AUTO_INCREMENT primary key column and designate it as the first column
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))       
        elif db_operation.lower() == "update": # if the operation is update
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append') # append the df to the end of the existing table with table_name

    else: # covers if an invalid operations specified
        print("'db_operation' must be either 'insert' or 'update'. Please rewrite function call.")
    
    db_connection.close()

"""
Name --> upsert_dataframe
Purpose --> updates or expands information in the target_df using information from source_df
    source_df: DataFrame, the data to update or expand the target_df with
    source_key: text string, the primary key of the source_df
    target_df: DataFrame, the data structure to be updated or expanded
    target_key: text string, the primary key of the target_df
Returns --> a dataframe with information in source_df either updating (i.e. replacing) or expanding information in the target_df
"""
def upsert_dataframe(source_df, source_key, target_df, target_key):
    #temporarily set index of source dataframe and append target dataframe to match the primary key of the source dataframe
    target_df.set_index(source_key, inplace=True)
    source_df.set_index(source_key, inplace=True)
    #match rows across both tables by index; if an index exists in both tables, update the target dataframe with corresponding values from the source
    target_df.update(source_df)
    #append any rows not present in the target dataframe but present in the source dataframe to the target dataframe, and sort by index
    df_upserted = target_df.combine_first(source_df).sort_index()
    #reset index to its place as a primary key column
    df_upserted.reset_index(level=None,inplace=True)
    #drop duplicate key
    df_upserted.drop(target_key,axis=1,inplace=True)
    return df_upserted

"""
Name --> get_mongo_client
Purpose --> to connect to a MongoDB cluster client
    user_name: text string, the user name for the host MongoDB account
    pwd: text string, the password corresponding to the user_name
    cluster_name: text string, the name of the cluster in MongoDB
    cluster_location: text string, name of the location of the cluster in MongoDB
    cluster_subnet: text string, the unique subnet string given by MongoDB for the cluster
Returns --> MongoDB client connection
"""
def get_mongo_client(user_name,pwd,cluster_name,cluster_location,cluster_subnet):
    #ensure that location matches MongoDB
    if cluster_location not in ['atlas', 'local']: #if not appropriate
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.") #throw error message
    else: # otherwise, connect to MongoDB
        if cluster_location == "atlas": #if the connection is to an atlas, connect with parameters
            connection_string = f"mongodb+srv://{user_name}:{pwd}@"
            connection_string += f"{cluster_name}.{cluster_subnet}.mongodb.net"
            client = pymongo.MongoClient(connection_string, tlsCAFile=certifi.where())
        elif cluster_location == "local": #if the connection is local, connect locally
            client = pymongo.MongoClient("mongodb://localhost:27017/")
    return client #return client to use

"""
Name --> get_mongo_dataframe
Purpose --> to retrieve a data table from a MongoDB cluster
    mongo_client: text string, name of the client to connect to
    db_name: text string, name of the database on MongoDB to Extract from within the cluster
    collection: text string, name of the collection to Extract from within the database
    query: the query to perform
Returns --> a data table formatted as a data frame from the specified MongoDB client

"""
def get_mongo_dataframe(mongo_client, db_name, collection, query):
    db = mongo_client[db_name] #get the database from the cluster
    df = pd.DataFrame(list(db[collection].find(query))) #within that database, find the specified collection and run the query
    df.drop(['_id'], axis=1, inplace=True) #remove id column
    mongo_client.close()
    
    return df #return dataframe

"""
Name --> set_mongo_collections
Purpose --> uploads json files as a collection to a MongoDB database within a specified cluster for later analysis
    mongo_client: text string, name of the client to connect to
    db_name: text string, name of the database on MongoDB to write to within the client cluster
    data_directory: text string, the directory from which to Extract the data files
    json_files: array of text strings, the file names (including extensions) to Extract
Returns --> nothing
"""
def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name] #set the target database
    
    for file in json_files: #for every file to Extract
        db.drop_collection(file) #drop the file if it already exists
        json_file = os.path.join(data_directory, json_files[file]) #get full path to current file from directory, including file name and extension
        with open(json_file, 'r') as openfile: #open the file
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()



#### 2.4. Create Destination Database

The **adventureworks_dw** MySQL database will constitue the data mart required by **Deliverable 1**.

In [4]:
#create project SQL database
create_and_use_database(user_id,pwd,host_name,dst_dbname)


### 3. Extract-Transform-Load Pipeline: SQL Database Source

The following code cells *extract* data from the **adventureworks** MySQL database created by the provided *AdventureWorks_MySQL* .sql script and  *transform* the data as needed to fit the desired format.

#### 3.1 Extract and Transform Customer-Related Data From Source SQL Database

##### 3.1.1. Extract and Transform Customer Data

In [5]:
sql_customer = f"SELECT * FROM `{src_dbname}`.customer;"
df_customer = get_dataframe(user_id,pwd,host_name,src_dbname,sql_customer) #get table from source database using SQL query
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #columns unneeded
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType
0,1,1,AW00000001,S
1,2,1,AW00000002,S


##### 3.1.2. Extract and Transform Customer Address Data and Merge to Customer Data

In [6]:
sql_customeraddress = f"SELECT * FROM `{src_dbname}`.customeraddress;"
df_customeraddress = get_dataframe(user_id,pwd,host_name,src_dbname,sql_customeraddress) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_customeraddress,how='inner',on='CustomerID') 
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID
0,1,1,AW00000001,S,832,3
1,2,1,AW00000002,S,297,5


##### 3.1.3. Extract and Transform Address Data and Merge to Customer Data

In [7]:
sql_address = f"SELECT * FROM `{src_dbname}`.address;"
df_address = get_dataframe(user_id,pwd,host_name,src_dbname,sql_address) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_address,how='inner',on='AddressID') 
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode
0,1,1,AW00000001,S,832,3,2251 Elliot Avenue,None,Seattle,79,98104
1,2,1,AW00000002,S,297,5,7943 Walnut Ave,None,Renton,79,98055


##### 3.1.4. Extract and Transform Address Type Data and Merge to Customer Data

In [8]:
sql_addresstype = f"SELECT * FROM `{src_dbname}`.addresstype;"
df_addresstype = get_dataframe(user_id,pwd,host_name,src_dbname,sql_addresstype) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_addresstype,how='inner',on='AddressTypeID')
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.rename(columns={'Name':'AddressType'},inplace=True) #rename column to avoid ambiguity with other name attributes down the line
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode,AddressType
0,1,1,AW00000001,S,832,3,2251 Elliot Avenue,None,Seattle,79,98104,Main Office
1,2,1,AW00000002,S,297,5,7943 Walnut Ave,None,Renton,79,98055,Shipping


##### 3.1.5. Extract and Transform State/Province Data and Merge to Customer Data

In [9]:
sql_stateprovince = f"SELECT * FROM `{src_dbname}`.stateprovince;"
df_stateprovince = get_dataframe(user_id,pwd,host_name,src_dbname,sql_stateprovince) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_stateprovince.drop('TerritoryID',axis=1),how='inner',on='StateProvinceID') #avoid duplication of TerritoryID
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.rename(columns={'Name':'StateProvince'},inplace=True) #rename column to avoid ambiguity with other name attributes down the line
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode,AddressType,StateProvinceCode,CountryRegionCode,IsOnlyStateProvinceFlag,StateProvince
0,1,1,AW00000001,S,832,3,2251 Elliot Avenue,None,Seattle,79,98104,Main Office,WA,US,b'\x00',Washington
1,2,1,AW00000002,S,297,5,7943 Walnut Ave,None,Renton,79,98055,Shipping,WA,US,b'\x00',Washington


##### 3.1.6. Extract and Transform Country Data and Merge to Customer Data

In [10]:
sql_countryregion = f"SELECT * FROM `{src_dbname}`.countryregion;"
df_countryregion = get_dataframe(user_id,pwd,host_name,src_dbname,sql_countryregion) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_countryregion,how='inner',on='CountryRegionCode')
df_customer.drop(['ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.rename(columns={'Name':'CountryRegion'},inplace=True) #rename column to avoid ambiguity with other name attributes down the line
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode,AddressType,StateProvinceCode,CountryRegionCode,IsOnlyStateProvinceFlag,StateProvince,CountryRegion
0,1,1,AW00000001,S,832,3,2251 Elliot Avenue,None,Seattle,79,98104,Main Office,WA,US,b'\x00',Washington,United States
1,2,1,AW00000002,S,297,5,7943 Walnut Ave,None,Renton,79,98055,Shipping,WA,US,b'\x00',Washington,United States


##### 3.1.7. Extract and Transform Sales Territory Data and Merge to Customer Data

In [11]:
sql_salesterritory = f"SELECT * FROM `{src_dbname}`.salesterritory;"
df_salesterritory = get_dataframe(user_id,pwd,host_name,src_dbname,sql_salesterritory) #get table from source database using SQL query
df_customer = pd.merge(df_customer,df_salesterritory.drop('CountryRegionCode',axis=1),how='inner',on='TerritoryID') #avoid duplication of CountryRegionCode
df_customer.drop(['rowguid','ModifiedDate'],axis=1,inplace=True) #drop unneeded columns
df_customer.rename(columns={'Name':'SalesTerritory','Group':'SalesTerritoryGroup'},inplace=True) #rename columns to avoid ambiguity with other name attributes down the line
df_customer.head(2)

,CustomerID,TerritoryID,AccountNumber,CustomerType,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,...,CountryRegionCode,IsOnlyStateProvinceFlag,StateProvince,CountryRegion,SalesTerritory,SalesTerritoryGroup,SalesYTD,SalesLastYear,CostYTD,CostLastYear
0,1,1,AW00000001,S,832,3,2251 Elliot Avenue,None,Seattle,79,...,US,b'\x00',Washington,United States,Northwest,North America,5.767342e+06,3.298694e+06,0.0,0.0
1,2,1,AW00000002,S,297,5,7943 Walnut Ave,None,Renton,79,...,US,b'\x00',Washington,United States,Northwest,North America,5.767342e+06,3.298694e+06,0.0,0.0


##### 3.1.8. Filter Customer Data to Desired Columns in Desired Order

In [12]:
df_customer = df_customer[['CustomerID','AccountNumber','CustomerType', 'City','PostalCode','AddressType',
                           'StateProvince','CountryRegion','SalesTerritory','SalesTerritoryGroup']] #filter for columns that best fit the analysis at hand
df_customer.head(2)

,CustomerID,AccountNumber,CustomerType,City,PostalCode,AddressType,StateProvince,CountryRegion,SalesTerritory,SalesTerritoryGroup
0,1,AW00000001,S,Seattle,98104,Main Office,Washington,United States,Northwest,North America
1,2,AW00000002,S,Renton,98055,Shipping,Washington,United States,Northwest,North America


#### 3.2 Extract and Transform Vendor-Related Data From Source SQL Database

##### 3.2.1. Extract and Transform Vendor Data

In [13]:
sql_vendor = f"SELECT * FROM `{src_dbname}`.vendor;"
df_vendor = get_dataframe(user_id,pwd,host_name,src_dbname,sql_vendor) #get table from source database using SQL query
df_vendor.drop(['ModifiedDate'],axis=1,inplace=True)  #drop unneeded columns
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',None
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',None


##### 3.2.2. Extract and Transform Vendor Address Data and Merge to Vendor Data

In [14]:
sql_vendoraddress = f"SELECT * FROM `{src_dbname}`.vendoraddress;"
df_vendoraddress = get_dataframe(user_id,pwd,host_name,src_dbname,sql_vendoraddress) #get table from source database using SQL query
df_vendoraddress.drop(['ModifiedDate'],axis=1,inplace=True)  #drop unneeded columns
df_vendor = pd.merge(df_vendor,df_vendoraddress,how='inner',on='VendorID')
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL,AddressID,AddressTypeID
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',None,357,3
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',None,335,3


##### 3.2.3. Extract and Transform Address Data and Merge to Vendor Data

In [15]:
sql_address = f"SELECT * FROM `{src_dbname}`.address;"
df_address = get_dataframe(user_id,pwd,host_name,src_dbname,sql_address) #get table from source database using SQL query
df_address.drop(['ModifiedDate','rowguid'],axis=1,inplace=True)  #drop unneeded columns
df_vendor = pd.merge(df_vendor,df_address,how='inner',on='AddressID')
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',None,357,3,683 Larch Ct.,None,Salt Lake City,74,84101
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',None,335,3,8547 Catherine Way,None,Tacoma,79,98403


##### 3.2.4. Extract and Transform State/Province Data and Merge to Vendor Data

In [16]:
sql_stateprovince = f"SELECT * FROM `{src_dbname}`.stateprovince;"
df_stateprovince = get_dataframe(user_id,pwd,host_name,src_dbname,sql_stateprovince) #get table from source database using SQL query
df_stateprovince.drop(['ModifiedDate','rowguid'],axis=1,inplace=True)  #drop unneeded columns
df_stateprovince.rename(columns={'Name':'StateProvince'},inplace=True) #rename columns to avoid ambiguity with other name attributes down the line
df_vendor = pd.merge(df_vendor,df_stateprovince,how='inner',on='StateProvinceID')
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode,StateProvinceCode,CountryRegionCode,IsOnlyStateProvinceFlag,StateProvince,TerritoryID
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',None,357,3,683 Larch Ct.,None,Salt Lake City,74,84101,UT,US,b'\x00',Utah,1
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',None,335,3,8547 Catherine Way,None,Tacoma,79,98403,WA,US,b'\x00',Washington,1


##### 3.2.5. Extract and Transform Address Type Data and Merge to Vendor Data

In [17]:
sql_addresstype = f"SELECT * FROM `{src_dbname}`.addresstype;"
df_addresstype =  get_dataframe(user_id,pwd,host_name,src_dbname,sql_addresstype) #get table from source database using SQL query
df_addresstype.drop(['ModifiedDate','rowguid'],axis=1,inplace=True)  #drop unneeded columns
df_addresstype.rename(columns={'Name':'AddressType'},inplace=True) #rename columns to avoid ambiguity with other name attributes down the line
df_vendor = pd.merge(df_vendor,df_addresstype,how='inner',on='AddressTypeID')
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,PurchasingWebServiceURL,AddressID,AddressTypeID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode,StateProvinceCode,CountryRegionCode,IsOnlyStateProvinceFlag,StateProvince,TerritoryID,AddressType
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',None,357,3,683 Larch Ct.,None,Salt Lake City,74,84101,UT,US,b'\x00',Utah,1,Main Office
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',None,335,3,8547 Catherine Way,None,Tacoma,79,98403,WA,US,b'\x00',Washington,1,Main Office


##### 3.2.6. Filter Vendor Data to Desired Columns in Desired Order

In [18]:
df_vendor = df_vendor[['VendorID','AccountNumber','Name','CreditRating','PreferredVendorStatus',
                       'ActiveFlag','AddressType','City','StateProvince','PostalCode']] #filter for columns that best fit the analysis at hand
df_vendor.head(2)

,VendorID,AccountNumber,Name,CreditRating,PreferredVendorStatus,ActiveFlag,AddressType,City,StateProvince,PostalCode
0,1,INTERNAT0001,International,1,b'\x01',b'\x01',Main Office,Salt Lake City,Utah,84101
1,2,ELECTRON0002,Electronic Bike Repair & Supplies,1,b'\x01',b'\x01',Main Office,Tacoma,Washington,98403


#### 3.3. Extract and Transform Address-Related Data From Source SQL Database

##### 3.3.1. Extract and Transform Address Data

In [19]:
sql_address = f"SELECT * FROM `{src_dbname}`.address;"
df_address = get_dataframe(user_id,pwd,host_name,src_dbname,sql_address) #get table from source database using SQL query
df_address.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)  #drop unneeded columns
df_address.head(2)

,AddressID,AddressLine1,AddressLine2,City,StateProvinceID,PostalCode
0,1,1970 Napa Ct.,None,Bothell,79,98011
1,2,9833 Mt. Dias Blv.,None,Bothell,79,98011


##### 3.3.2. Extract and Transform State/Province Data and Merge to Address Data

In [20]:
sql_stateprovince = f"SELECT * FROM `{src_dbname}`.stateprovince;"
df_stateprovince = get_dataframe(user_id,pwd,host_name,src_dbname,sql_stateprovince) #get table from source database using SQL query
df_stateprovince.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)  #drop unneeded columns
df_stateprovince.rename(columns={'Name':'StateProvince'},inplace=True) #rename columns to avoid ambiguity with other name attributes down the line
df_address = pd.merge(df_address,df_stateprovince[['StateProvinceID','StateProvince']],how='inner',on='StateProvinceID')

df_address = df_address[['AddressID','AddressLine1','AddressLine2','City','StateProvince','PostalCode']] #filter for columns that best fit the analysis at hand
df_address.head(2)

,AddressID,AddressLine1,AddressLine2,City,StateProvince,PostalCode
0,1,1970 Napa Ct.,None,Bothell,Washington,98011
1,2,9833 Mt. Dias Blv.,None,Bothell,Washington,98011


#### 3.4. Extract and Transform Ship Method-Related Data From Source SQL Database

In [21]:
sql_shipmethod = f"SELECT * FROM `{src_dbname}`.shipmethod;"
df_shipmethod = get_dataframe(user_id,pwd,host_name,src_dbname,sql_shipmethod) #get table from source database using SQL query
df_shipmethod.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)  #drop unneeded columns
df_shipmethod.head(2)

,ShipMethodID,Name,ShipBase,ShipRate
0,1,XRQ - TRUCK GROUND,3.95,0.99
1,2,ZY - EXPRESS,9.95,1.99


### 4. Extract-Transform-Load Pipeline: Local File System Source

The following code cells *extract* data from a .json file created by exporting a view generated by the provided *AdventureWorks_Queries_MySQL* .sql script and  *transform* the data as needed to fit the desired format. Please change the file path as necessary for the local machine. The extension shows .csv as result of the MySQL export.

#### 4.1. Extract Employee-Related Data From Local File

In [22]:
os.chdir("C:/Users/dinos/OneDrive/Desktop/College/8th Semester/DS 2002 - Data Science Systems/Project 1") #set working directory, change
json_file = "adventureworks_dim_employee_view.csv"
df_employee = pd.read_json(json_file) #read in file


#### 4.2. Transform Employee-Related Data From Local File

In [23]:
df_employee.rename(columns={'Title':'JobTitle'},inplace=True) #rename to avoid later ambiguities
df_employee = df_employee[['EmployeeID','NationalIDNumber','ManagerID','JobTitle','FirstName',
                           'MiddleName','LastName','EmailAddress','Phone','CurrentFlag']] #select columns of interest and reorder
df_employee.head(2)

,EmployeeID,NationalIDNumber,ManagerID,JobTitle,FirstName,MiddleName,LastName,EmailAddress,Phone,CurrentFlag
0,1,14417807,16,Production Technician - WC60,Guy,R,Gilbert,guy1@adventure-works.com,320-555-0195,1
1,2,253022876,6,Marketing Assistant,Kevin,F,Brown,kevin0@adventure-works.com,150-555-0189,1


### 5. Extract-Transform-Load Pipeline: NoSQL Database Source

The following code cells *extract* data from a NoSQL database by populating a MongoDB cluster with .json files generated as in Item 4 and  *transform* the data as needed to fit the desired format. 

#### 5.1. Populate Product-Related Data Into MongoDB Cluster

In [24]:
client = get_mongo_client(mongo_user_name,mongo_pwd,cluster_name,cluster_location,cluster_subnet)

data_directory = os.getcwd()

json_files = {'product':'adventureworks_dim_product_view.csv'}

set_mongo_collections(client,mongo_db_name,data_directory,json_files)

#### 5.2. Extract and Transform Product-Related Data From MongoDB Cluster

In [25]:
client = get_mongo_client(mongo_user_name,mongo_pwd,cluster_name,cluster_location,cluster_subnet) #get mongo client to use
query = {} #get all data
collection = 'product' #set collection name
df_product = get_mongo_dataframe(client,mongo_db_name,collection,query) #get data from cluster
df_product.drop(['DiscontinuedDate'],axis=1,inplace=True) #drop unnecessary column
df_product.head(5)

,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,...,Weight,DaysToManufacture,ProductLine,Class,Style,ProductCategory,ProductSubcategory,ProductModel,SellStartDate,SellEndDate
0,1,Adjustable Race,AR-5381,0,0,,1000,750,0.0,0.0,...,0.0,0,,,,,,,1998-06-01 00:00:00,
1,2,Bearing Ball,BA-8327,0,0,,1000,750,0.0,0.0,...,0.0,0,,,,,,,1998-06-01 00:00:00,
2,3,BB Ball Bearing,BE-2349,1,0,,800,600,0.0,0.0,...,0.0,1,,,,,,,1998-06-01 00:00:00,
3,4,Headset Ball Bearings,BE-2908,0,0,,800,600,0.0,0.0,...,0.0,0,,,,,,,1998-06-01 00:00:00,
4,316,Blade,BL-2036,1,0,,800,600,0.0,0.0,...,0.0,1,,,,,,,1998-06-01 00:00:00,


### 6. Extract-Transform-Load Pipeline: Load Dimension Data into Database

The following code cells *load* the extracted and transformed data into corresponding dimension tables in the **adventureworks_dw** database (with the exception of *dim_date*, which must be made by running the *Create_Populate_Dim_Date.sql script*).

In [26]:
#tuples are arranged into table_name, df, pk_column inputs for the set_dataframe function
tables = [('dim_product',df_product,'ProductKey'),
          ('dim_customer',df_customer,'CustomerKey'),
          ('dim_employee',df_employee,'EmployeeKey'),
          ('dim_vendor',df_vendor,'VendorKey'),
          ('dim_shipmethod',df_shipmethod,'ShipMethodKey'),
          ('dim_address',df_address,'AddressKey')]

for name, df, pk in tables:
    set_dataframe(user_id,pwd,host_name,dst_dbname,df,name,pk,'insert')

#### 7. Extract-Transform-Load Pipeline: Fact Table Creation

Similarly to previous sections, the following cells *extract, transform, and load* relevant data from the **adventureworks** database to create two fact tables: *fact_purchase_orders* and *fact_sales_order_lines*. The following cells also create the key relationships to each of the dimension tables.

##### 7.1. Extract and Transform Purchase Order Header-Related Data 

In [36]:
sql_purchaseorderheader = f"SELECT * FROM `{src_dbname}`.purchaseorderheader;"
df_purchaseorderheader = get_dataframe(user_id,pwd,host_name,src_dbname,sql_purchaseorderheader)
df_purchaseorderheader.drop(['ModifiedDate'],axis=1,inplace=True)
df_purchaseorderheader.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ShipMethodID,OrderDate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue
0,1,0,4,244,83,3,2001-05-17,2001-05-26,201.0400,16.0832,5.0260,222.1492
1,2,0,1,231,32,5,2001-05-17,2001-05-26,272.1015,21.7681,6.8025,300.6721


##### 7.2. Extract and Transform Ship Method-Related Data and Merge to Purchase Order Header Data to Create fact_purchase_orders

In [37]:
sql_shipmethod = f"SELECT * FROM `{src_dbname}`.shipmethod;"
df_shipmethod = get_dataframe(user_id,pwd,host_name,src_dbname,sql_shipmethod)
df_shipmethod.rename(columns={'Name':'ShipMethod'},inplace=True)
df_shipmethod.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)
df_fact_purchase_orders = pd.merge(df_purchaseorderheader,df_shipmethod,how='inner',on='ShipMethodID')
df_fact_purchase_orders.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ShipMethodID,OrderDate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,ShipMethod,ShipBase,ShipRate
0,1,0,4,244,83,3,2001-05-17,2001-05-26,201.0400,16.0832,5.0260,222.1492,OVERSEAS - DELUXE,29.95,2.99
1,2,0,1,231,32,5,2001-05-17,2001-05-26,272.1015,21.7681,6.8025,300.6721,CARGO TRANSPORT 5,8.99,1.49


##### 7.3. Extract and Transform Purchase Order Detail-Related Data and Merge to fact_purchase_orders

In [38]:
sql_purchaseorderdetail = f"SELECT * FROM `{src_dbname}`.purchaseorderdetail;"
df_purchaseorderdetail = get_dataframe(user_id,pwd,host_name,src_dbname,sql_purchaseorderdetail)
df_purchaseorderdetail.drop(['ModifiedDate'],axis=1,inplace=True)
df_fact_purchase_orders = pd.merge(df_fact_purchase_orders,df_purchaseorderdetail,how='left',on='PurchaseOrderID')
df_fact_purchase_orders.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ShipMethodID,OrderDate,ShipDate,SubTotal,TaxAmt,...,ShipRate,PurchaseOrderDetailID,DueDate,OrderQty,ProductID,UnitPrice,LineTotal,ReceivedQty,RejectedQty,StockedQty
0,1,0,4,244,83,3,2001-05-17,2001-05-26,201.0400,16.0832,...,2.99,1,2001-05-31,4,1,50.26,201.04,3.0,0.0,3.0
1,2,0,1,231,32,5,2001-05-17,2001-05-26,272.1015,21.7681,...,1.49,2,2001-05-31,3,359,45.12,135.36,3.0,0.0,3.0


##### 7.4. Filter fact_purchase_orders to Desired Columns in Desired Order

In [39]:
df_fact_purchase_orders = df_fact_purchase_orders[['PurchaseOrderID','RevisionNumber','Status','EmployeeID', 'VendorID','ProductID', 'ShipMethodID',
                                                   'OrderDate', 'ShipDate', 'DueDate','OrderQty','UnitPrice', 'LineTotal','SubTotal','TaxAmt','Freight',
                                                   'TotalDue','ReceivedQty','RejectedQty','StockedQty']]
df_fact_purchase_orders.head(2)

,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,ShipMethodID,OrderDate,ShipDate,DueDate,OrderQty,UnitPrice,LineTotal,SubTotal,TaxAmt,Freight,TotalDue,ReceivedQty,RejectedQty,StockedQty
0,1,0,4,244,83,1,3,2001-05-17,2001-05-26,2001-05-31,4,50.26,201.04,201.0400,16.0832,5.0260,222.1492,3.0,0.0,3.0
1,2,0,1,231,32,359,5,2001-05-17,2001-05-26,2001-05-31,3,45.12,135.36,272.1015,21.7681,6.8025,300.6721,3.0,0.0,3.0


#### 7.5. Replace Business Keys in fact_purchase_orders With Corresponding Surrogate Primary Keys

##### 7.5.1. Get Surrogate Keys and Business Keys for Each Dimension

Please run the Create_Populate_Dim_Date.sql script before running any cells past this point, otherwise errors will occur.

In [40]:
sql_dim_customer = f"SELECT CustomerKey, CustomerID FROM `{dst_dbname}`.dim_customer;"
df_dim_customer = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_customer)

sql_dim_employee = f"SELECT EmployeeKey, EmployeeID FROM `{dst_dbname}`.dim_employee;"
df_dim_employee = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_employee)

sql_dim_product = f"SELECT ProductKey, ProductID FROM `{dst_dbname}`.dim_product;"
df_dim_product = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_product)

sql_dim_vendor = f"SELECT VendorKey, VendorID FROM `{dst_dbname}`.dim_vendor;"
df_dim_vendor = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_vendor)

sql_dim_shipmethod = f"SELECT ShipMethodKey, ShipMethodID FROM `{dst_dbname}`.dim_shipmethod;"
df_dim_shipmethod = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_shipmethod)

#general date dimension
sql_dim_date = f"SELECT date_key, full_date from `{dst_dbname}`.dim_date;"
df_dim_date = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_date)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date

#date dimension taking on specific roles
df_dim_order_date = df_dim_date.rename(columns={'date_key':'OrderDateKey','full_date':'OrderDate'})
df_fact_purchase_orders.OrderDate = df_fact_purchase_orders.OrderDate.astype('datetime64[ns]').dt.date

df_dim_ship_date = df_dim_date.rename(columns={'date_key':'ShipDateKey','full_date':'ShipDate'})
df_fact_purchase_orders.ShipDate = df_fact_purchase_orders.ShipDate.astype('datetime64[ns]').dt.date

df_dim_due_date = df_dim_date.rename(columns={'date_key':'DueDateKey','full_date':'DueDate'})
df_fact_purchase_orders.DueDate = df_fact_purchase_orders.DueDate.astype('datetime64[ns]').dt.date


##### 7.5.2. Replace Business Keys With Surrogate Keys

In [41]:
#using the business keys and dimension tables, lookup corresponding surrogate primary keys and add to fact table
dimensions = [('EmployeeID',df_dim_employee),
              ('VendorID',df_dim_vendor),
              ('ProductID',df_dim_product),
              ('ShipMethodID',df_dim_shipmethod),
              ('OrderDate',df_dim_order_date),
              ('ShipDate',df_dim_ship_date),
              ('DueDate',df_dim_due_date)]

for business_key, dimension_table in dimensions:
    df_fact_purchase_orders = pd.merge(df_fact_purchase_orders, dimension_table, on=business_key,how='inner')
    df_fact_purchase_orders.drop(business_key,axis=1,inplace=True)

df_fact_purchase_orders = df_fact_purchase_orders[['PurchaseOrderID','RevisionNumber','EmployeeKey', 'VendorKey','ProductKey', 'ShipMethodKey',
                                                   'OrderDateKey', 'ShipDateKey', 'DueDateKey','OrderQty','UnitPrice', 'LineTotal','SubTotal','TaxAmt','Freight',
                                                   'TotalDue','ReceivedQty','RejectedQty','StockedQty']]

df_fact_purchase_orders.head(2)


,PurchaseOrderID,RevisionNumber,EmployeeKey,VendorKey,ProductKey,ShipMethodKey,OrderDateKey,ShipDateKey,DueDateKey,OrderQty,UnitPrice,LineTotal,SubTotal,TaxAmt,Freight,TotalDue,ReceivedQty,RejectedQty,StockedQty
0,1,0,257,83,1,3,20010517,20010526,20010531,4,50.26,201.04,201.0400,16.0832,5.0260,222.1492,3.0,0.0,3.0
1,2,0,233,32,159,5,20010517,20010526,20010531,3,45.12,135.36,272.1015,21.7681,6.8025,300.6721,3.0,0.0,3.0


##### 7.6. Extract and Transform Sales Order Header-Related Data 

In [42]:
sql_salesorderheader = f"SELECT * FROM `{src_dbname}`.salesorderheader;"
df_salesorderheader = get_dataframe(user_id,pwd,host_name,src_dbname,sql_salesorderheader)
df_salesorderheader.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)
df_salesorderheader.head(2)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,ShipToAddressID,ShipMethodID,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,985,5,16281.0,105041Vi84182,NaN,24643.9362,1971.5149,616.0984,27231.5495,None
1,43660,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43660,PO18850127500,10-4020-000117,...,921,5,5618.0,115213Vi29411,NaN,1553.1035,124.2483,38.8276,1716.1794,None


##### 7.7. Extract and Transform Sales Order Detail-Related Data and Merge to Sales Order Header Data to create fact_sales_order_lines

In [43]:
sql_salesorderdetail = f"SELECT * FROM `{src_dbname}`.salesorderdetail;"
df_salesorderdetail = get_dataframe(user_id,pwd,host_name,src_dbname,sql_salesorderdetail)
df_salesorderdetail.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)
df_fact_sales_order_lines = pd.merge(df_salesorderheader,df_salesorderdetail,how='left',on='SalesOrderID')
df_fact_sales_order_lines.head(2)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,TotalDue,Comment,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,27231.5495,None,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994
1,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,27231.5495,None,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982


#### 7.8 Extract and Transform Sales Territory-Related Data and Merge to fact_sales_order_lines

In [44]:
sql_salesterritory = f"SELECT * FROM `{src_dbname}`.salesterritory;"
df_salesterritory = get_dataframe(user_id,pwd,host_name,src_dbname,sql_salesterritory)
df_salesterritory.drop(['rowguid','ModifiedDate'],axis=1,inplace=True)
df_salesterritory.rename(columns={'Name':'SalesTerritory','Group':'SalesTerritoryGroup'},inplace=True)
df_fact_sales_order_lines = pd.merge(df_fact_sales_order_lines,df_salesterritory,how='inner',on='TerritoryID')
df_fact_sales_order_lines.head(2)

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,UnitPrice,UnitPriceDiscount,LineTotal,SalesTerritory,CountryRegionCode,SalesTerritoryGroup,SalesYTD,SalesLastYear,CostYTD,CostLastYear
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,2024.994,0.0,2024.994,Southeast,US,North America,2.851419e+06,3.925071e+06,0.0,0.0
1,43659,1,2001-07-01,2001-07-13,2001-07-08,5,b'\x00',SO43659,PO522145787,10-4020-000676,...,2024.994,0.0,6074.982,Southeast,US,North America,2.851419e+06,3.925071e+06,0.0,0.0


#### 7.9. Replace Business Keys in fact_sales_order_lines With Corresponding Surrogate Primary Keys

##### 7.9.1. Get Surrogate Keys and Business Keys for Each Dimension

In [45]:
sql_dim_customer = f"SELECT CustomerKey, CustomerID FROM `{dst_dbname}`.dim_customer;"
df_dim_customer = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_customer)

sql_dim_employee = f"SELECT EmployeeKey, EmployeeID FROM `{dst_dbname}`.dim_employee;"
df_dim_employee = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_employee)
df_dim_employee.rename(columns={'EmployeeID':'SalesPersonID'},inplace=True) # SalesPersonID is an EmployeeID in the source database

sql_dim_product = f"SELECT ProductKey, ProductID FROM `{dst_dbname}`.dim_product;"
df_dim_product = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_product)

sql_dim_shipmethod = f"SELECT ShipMethodKey, ShipMethodID FROM `{dst_dbname}`.dim_shipmethod;"
df_dim_shipmethod = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_shipmethod)

#general date dimension
sql_dim_date = f"SELECT date_key, full_date from `{dst_dbname}`.dim_date;"
df_dim_date = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_date)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date

#date dimension taking on specific roles
df_dim_order_date = df_dim_date.rename(columns={'date_key':'OrderDateKey','full_date':'OrderDate'})
df_fact_sales_order_lines.OrderDate = df_fact_sales_order_lines.OrderDate.astype('datetime64[ns]').dt.date

# df_dim_ship_date = df_dim_date.rename(columns={'date_key':'ShipDateKey','full_date':'ShipDate'})
df_fact_sales_order_lines.ShipDate = df_fact_sales_order_lines.ShipDate.astype('datetime64[ns]').dt.date

df_dim_due_date = df_dim_date.rename(columns={'date_key':'DueDateKey','full_date':'DueDate'})
df_fact_sales_order_lines.DueDate = df_fact_sales_order_lines.DueDate.astype('datetime64[ns]').dt.date

#general address dimension
sql_dim_address = f"SELECT AddressKey, AddressID FROM `{dst_dbname}`.dim_address;"
df_dim_address = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_dim_address)

#address dimension taking on specific roles
df_dim_bill_to_address = df_dim_address.rename(columns={'AddressKey':'BillToAddressKey','AddressID':'BillToAddressID'})

df_dim_ship_to_address = df_dim_address.rename(columns={'AddressKey':'ShipToAddressKey','AddressID':'ShipToAddressID'})


##### 7.9.2. Replace Business Keys With Surrogate Keys

In [46]:
#using the business keys and dimension tables, lookup corresponding surrogate primary keys and add to fact table
dimensions = [('SalesPersonID',df_dim_employee),
              ('ProductID',df_dim_product),
              ('ShipMethodID',df_dim_shipmethod),
              ('CustomerID', df_dim_customer),
              ('OrderDate',df_dim_order_date),
              ('ShipDate',df_dim_ship_date),
              ('DueDate',df_dim_due_date),
              ('ShipToAddressID',df_dim_ship_to_address),
              ('BillToAddressID',df_dim_bill_to_address)]

for business_key, dimension_table in dimensions:
    df_fact_sales_order_lines = pd.merge(df_fact_sales_order_lines, dimension_table, on=business_key,how='inner')
    df_fact_sales_order_lines.drop(business_key,axis=1,inplace=True)

df_fact_sales_order_lines = df_fact_sales_order_lines[['SalesOrderID','AccountNumber','SalesOrderNumber','PurchaseOrderNumber','RevisionNumber','CarrierTrackingNumber','OnlineOrderFlag',
                                                        'OrderDateKey','DueDateKey','ShipDateKey','ProductKey','CustomerKey','EmployeeKey','ShipMethodKey','BillToAddressKey','ShipToAddressKey',
                                                        'SalesTerritoryGroup','SalesTerritory','OrderQty','UnitPrice','LineTotal','SubTotal','TaxAmt','Freight','TotalDue']]

df_fact_sales_order_lines.head(2)

,SalesOrderID,AccountNumber,SalesOrderNumber,PurchaseOrderNumber,RevisionNumber,CarrierTrackingNumber,OnlineOrderFlag,OrderDateKey,DueDateKey,ShipDateKey,...,ShipToAddressKey,SalesTerritoryGroup,SalesTerritory,OrderQty,UnitPrice,LineTotal,SubTotal,TaxAmt,Freight,TotalDue
0,43659,10-4020-000676,SO43659,PO522145787,1,4911-403C-98,b'\x00',20010701,20010713,20010708,...,1132,North America,Southeast,1,2024.994,2024.994,24643.9362,1971.5149,616.0984,27231.5495
1,43659,10-4020-000676,SO43659,PO522145787,1,4911-403C-98,b'\x00',20010701,20010713,20010708,...,1132,North America,Southeast,3,2024.994,6074.982,24643.9362,1971.5149,616.0984,27231.5495


##### 7.10. Load Fact Tables Into Database

The following cell loads both fact tables into **adventureworks_dw**.

In [47]:
set_dataframe(user_id,pwd,host_name,dst_dbname,df_fact_purchase_orders,'fact_purchase_orders','FactPurchaseOrderKey','insert')
set_dataframe(user_id,pwd,host_name,dst_dbname,df_fact_sales_order_lines,'fact_sales_order_lines','FactSalesOrderLinesKey','insert')

#### 8. Testing Proper Functionality With Queries
The following cell provides a query which tests the functionality of the **adventureworks_dw** solution.

In [48]:
sql_sales_by_territory = """
SELECT
    CONCAT(de.FirstName, ' ', de.LastName) AS `Employee Name`,
    dp.Name AS `Product Name`,
    fosl.SalesTerritory AS `Sales Territory`,
    COUNT(*) AS `Total Sales Lines`,
    SUM(fosl.OrderQty) AS `Total Order Quantity`
FROM `{db}`.fact_sales_order_lines fosl
LEFT JOIN `{db}`.dim_product dp
    ON fosl.ProductKey = dp.ProductKey
LEFT JOIN `{db}`.dim_employee de
    ON fosl.EmployeeKey = de.EmployeeKey
GROUP BY
    de.EmployeeKey,
    dp.ProductKey,
    fosl.SalesTerritory
;
""".format(db=dst_dbname)

In [49]:
df_sales_by_territory = get_dataframe(user_id,pwd,host_name,dst_dbname,sql_sales_by_territory)
df_sales_by_territory.head(30)

,Employee Name,Product Name,Sales Territory,Total Sales Lines,Total Order Quantity
0,Tsvi Reiter,"Mountain-100 Black, 42",Southeast,34,134.0
1,Tsvi Reiter,"Mountain-100 Black, 44",Southeast,39,137.0
2,Michael Blythe,"Road-250 Red, 44",Southwest,13,47.0
3,Tsvi Reiter,"Mountain-100 Black, 48",Southeast,36,108.0
4,Tsvi Reiter,"Mountain-100 Silver, 38",Southeast,36,120.0
5,Michael Blythe,"LL Road Frame - Red, 62",Southwest,12,27.0
6,Tsvi Reiter,"Mountain-100 Silver, 42",Southeast,32,124.0
7,Michael Blythe,"ML Road Frame-W - Yellow, 48",Southwest,25,67.0
8,Tsvi Reiter,"Mountain-100 Silver, 44",Southeast,37,140.0
9,Tsvi Reiter,"Mountain-100 Silver, 48",Southeast,36,126.0
